#Сравнительный анализ методов кластеризации для географической и экономической сегментации данных о жилье в Калифорнии

Objectives: выполняет задачу сегментации (кластеризации) районов Калифорнии на основе набора признаков, связанных с жильём и демографией. Используется датасет California Housing Dataset, который содержит информацию о средних ценах на жильё, количестве комнат, населении, доходах и географических координатах.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

data = fetch_california_housing(as_frame=True)
df = data.frame

df['Bedroom_Ratio'] = df['AveBedrms'] / df['AveRooms']
df['People_per_Household'] = df['Population'] / df['AveOccup']
df['Rooms_per_Person'] = df['AveRooms'] / df['People_per_Household']

print(f"Размер датасета: {df.shape}")
print(f"Количество столбцов: {df.shape[1]}")

df_sample = df.sample(n=2000, random_state=42).copy()

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_sample), columns=df_sample.columns)

geo_features = ['Latitude', 'Longitude']
other_features = [col for col in df_sample.columns if col not in geo_features]

X_all = df_scaled.values

X_no_geo = df_scaled[other_features].values

print("Данные подготовлены.")

Размер датасета: (20640, 12)
Количество столбцов: 12
Данные подготовлены.


In [3]:
def evaluate_clustering(X, model_name, model):

    labels = model.fit_predict(X)


    unique_labels = np.unique(labels)
    if len(unique_labels) < 2:
        return None, labels

    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)

    return {'Model': model_name, 'Silhouette': sil, 'Davies-Bouldin': db, 'Calinski-Harabasz': ch}, labels

results = []
best_labels_scenario_A = None
best_score_A = -1
best_labels_scenario_B = None
best_score_B = -1

In [4]:
print("--- Запуск Варианта А (С геоданными) ---")

kmeans_a = KMeans(n_clusters=5, random_state=42, n_init=10)
res_km, labels_km = evaluate_clustering(X_all, 'A_KMeans', kmeans_a)
results.append(res_km)

agg_a = AgglomerativeClustering(n_clusters=5)
res_agg, labels_agg = evaluate_clustering(X_all, 'A_Agglomerative', agg_a)
results.append(res_agg)

dbscan_a = DBSCAN(eps=2.0, min_samples=5)
res_db, labels_db = evaluate_clustering(X_all, 'A_DBSCAN', dbscan_a)
results.append(res_db)

best_labels_scenario_A = labels_km

pd.DataFrame([r for r in results if r is not None])

--- Запуск Варианта А (С геоданными) ---


,Model,Silhouette,Davies-Bouldin,Calinski-Harabasz
0,A_KMeans,0.240030,1.216037,378.553904
1,A_Agglomerative,0.217092,1.247117,318.427724
2,A_DBSCAN,0.558032,2.755889,106.988815


In [5]:
print("--- Запуск Варианта Б (БЕЗ геоданных) ---")

kmeans_b = KMeans(n_clusters=5, random_state=42, n_init=10)
res_km_b, labels_km_b = evaluate_clustering(X_no_geo, 'B_KMeans', kmeans_b)
results.append(res_km_b)

agg_b = AgglomerativeClustering(n_clusters=5)
res_agg_b, labels_agg_b = evaluate_clustering(X_no_geo, 'B_Agglomerative', agg_b)
results.append(res_agg_b)

dbscan_b = DBSCAN(eps=1.5, min_samples=5)
res_db_b, labels_db_b = evaluate_clustering(X_no_geo, 'B_DBSCAN', dbscan_b)
if res_db_b: results.append(res_db_b)

best_labels_scenario_B = labels_km_b

final_df = pd.DataFrame([r for r in results if r is not None]).sort_values(by='Silhouette', ascending=False)
display(final_df)

--- Запуск Варианта Б (БЕЗ геоданных) ---


,Model,Silhouette,Davies-Bouldin,Calinski-Harabasz
2,A_DBSCAN,0.558032,2.755889,106.988815
5,B_DBSCAN,0.293360,2.076525,44.732226
3,B_KMeans,0.256939,0.978494,448.153223
0,A_KMeans,0.240030,1.216037,378.553904
4,B_Agglomerative,0.219323,1.009786,370.335671
1,A_Agglomerative,0.217092,1.247117,318.427724


In [6]:
df_sample['Cluster_A'] = best_labels_scenario_A
df_sample['Cluster_B'] = best_labels_scenario_B

map_center = [df_sample['Latitude'].mean(), df_sample['Longitude'].mean()]

colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

def create_map(scenario_name, cluster_col):
    m = folium.Map(location=map_center, zoom_start=6)


    for idx, row in df_sample.iterrows():
        cluster_id = int(row[cluster_col])

        color = colors[cluster_id % len(colors)] if cluster_id != -1 else 'black'

        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=3,
            color=color,
            fill=True,
            fill_color=color,
            popup=f"Price: {row['MedHouseVal']:.1f}"
        ).add_to(m)

    print(f"Карта для сценария: {scenario_name}")
    return m

map_a = create_map("А (Учитывая координаты)", 'Cluster_A')
display(map_a)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
map_b = create_map("Б (Только экономические признаки)", 'Cluster_B')
display(map_b)

Output hidden; open in https://colab.research.google.com to view.

#Conclusion

Вариант А (С геоданными):

- При использовании координат (Latitude, Longitude) кластеризация носит географический характер.

- На карте А вы увидите четкие зоны: кластер Сан-Франциско, кластер Лос-Анджелеса, кластер глубинки штата.

- Алгоритмы группируют объекты, которые находятся физически близко друг к другу. Метрика Silhouette обычно выше, так как координаты создают явные плотные группы.

Вариант Б (Без геоданных):

- Кластеризация группирует объекты по похожести характеристик (цена, количество комнат, население).

- На карте Б точки одного цвета разбросаны по всему штату (mixed distribution).

- Например, "богатый район" (один кластер) может появиться и на севере, и на юге. Это позволяет выделить типы недвижимости независимо от их местоположения.

Сравнение моделей:

- K-Means: Хорошо работает как базовый метод, но стремится создавать кластеры одинакового размера, что не всегда верно для географии.

- DBSCAN: Лучше всего подходит для геоданных (Вариант А), так как может выделять области произвольной формы (например, линию побережья) и отбрасывать выбросы (шум).

- Agglomerative: Дает похожие на KMeans результаты, но позволяет лучше понять иерархию (если строить дендрограмму).